In [2]:
import cv2
import torch
import numpy as np
import os
import time
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

In [7]:
# Modify these paths as needed
VIDEO_PATH = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/highway.avi"
OUTPUT_PATH = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/output.mp4"

CONF_THRESHOLD = 0.50
CLASS_ID = None       # set integer to track specific class
BLUR_CLASS_ID = None  # set integer to blur specific class

COCO_LABELS_PATH = "configs/coco.names"


In [8]:
def draw_corner_rect(img, bbox, line_length=30, line_thickness=5, rect_thickness=1,
                     rect_color=(255, 0, 255), line_color=(0, 255, 0)):
    x, y, w, h = bbox
    x1, y1 = x + w, y + h

    if rect_thickness != 0:
        cv2.rectangle(img, bbox, rect_color, rect_thickness)

    cv2.line(img, (x, y), (x + line_length, y), line_color, line_thickness)
    cv2.line(img, (x, y), (x, y + line_length), line_color, line_thickness)

    cv2.line(img, (x1, y), (x1 - line_length, y), line_color, line_thickness)
    cv2.line(img, (x1, y), (x1, y + line_length), line_color, line_thickness)

    cv2.line(img, (x, y1), (x + line_length, y1), line_color, line_thickness)
    cv2.line(img, (x, y1), (x, y1 - line_length), line_color, line_thickness)

    cv2.line(img, (x1, y1), (x1 - line_length, y1), line_color, line_thickness)
    cv2.line(img, (x1, y1), (x1, y1 - line_length), line_color, line_thickness)

    return img

def calculate_speed(distance, fps):
    return (distance * fps) * 3.6

def calculate_distance(p1, p2):
    return np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

def read_frames(cap):
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        yield frame


In [9]:
FRAME_WIDTH = 30
FRAME_HEIGHT = 100

SOURCE_POLYGON = np.array([[20, 200], [300, 220], [280, 100], [40, 80]], dtype=np.float32)
BIRD_EYE_VIEW = np.array([[0, 0], [FRAME_WIDTH, 0], [FRAME_WIDTH, FRAME_HEIGHT], [0, FRAME_HEIGHT]], dtype=np.float32)

M = cv2.getPerspectiveTransform(SOURCE_POLYGON, BIRD_EYE_VIEW)


In [10]:
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "Error opening video!"

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Polygon mask
pts = SOURCE_POLYGON.astype(np.int32).reshape((-1, 1, 2))
polygon_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)
cv2.fillPoly(polygon_mask, [pts], 255)

# Output video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (frame_width, frame_height))

# DeepSort tracker
tracker = DeepSort(max_age=50)

# YOLO model
model = YOLO("yolov10n.pt")

# COCO labels
with open(COCO_LABELS_PATH) as f:
    class_names = f.read().strip().split("\n")

# Colors
colors = np.random.randint(0, 255, (len(class_names), 3))

# State
prev_positions = {}
speed_accumulator = {}
frame_generator = read_frames(cap)

frame_count = 0
start_time = time.time()


/Users/veeralpatel/vehicle-speed-estimation-dip/.conda/lib/python3.11/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [11]:
for frame in frame_generator:

    # YOLO inference
    with torch.no_grad():
        results = model(frame)

    detections = []
    for pred in results:
        for box in pred.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = float(box.conf[0])
            label = int(box.cls[0])

            if confidence < CONF_THRESHOLD:
                continue

            if CLASS_ID is not None and label != CLASS_ID:
                continue

            if polygon_mask[(y1+y2)//2, (x1+x2)//2] == 255:
                detections.append([[x1, y1, x2-x1, y2-y1], confidence, label])

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue

        tid = track.track_id
        x1, y1, x2, y2 = map(int, track.to_ltrb())
        class_id = track.get_det_class()

        if polygon_mask[(y1+y2)//2, (x1+x2)//2] == 0:
            continue

        color = colors[class_id]

        # Compute center point
        center_pt = np.array([[(x1+x2)//2, (y1+y2)//2]], dtype=np.float32)
        transformed_pt = cv2.perspectiveTransform(center_pt[None,:,:], M)[0][0]

        # Speed calc
        if tid in prev_positions:
            distance = calculate_distance(prev_positions[tid], transformed_pt)
            speed = calculate_speed(distance, fps)

            if tid not in speed_accumulator:
                speed_accumulator[tid] = []
            speed_accumulator[tid].append(speed)
            speed_accumulator[tid] = speed_accumulator[tid][-100:]

        prev_positions[tid] = transformed_pt

        # Draw boxes
        frame = draw_corner_rect(frame, (x1, y1, x2-x1, y2-y1), 
                                 line_length=15, line_thickness=3,
                                 rect_thickness=1,
                                 rect_color=tuple(map(int,color)),
                                 line_color=tuple(map(int,color[::-1])))

        label_text = f"{tid} - {class_names[class_id]}"
        cv2.putText(frame, label_text, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        # Draw speed
        if tid in speed_accumulator:
            avg_speed = sum(speed_accumulator[tid]) / len(speed_accumulator[tid])
            cv2.putText(frame, f"Speed: {avg_speed:.1f} km/h",
                        (x1, y1-30), cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0,0,255), 2)

        # Gaussian blur if needed
        if BLUR_CLASS_ID is not None and class_id == BLUR_CLASS_ID:
            frame[y1:y2, x1:x2] = cv2.GaussianBlur(frame[y1:y2, x1:x2], (99,99), 3)

    cv2.polylines(frame, [pts], True, (255,0,0), 2)
    writer.write(frame)

    frame_count += 1
    if frame_count % 10 == 0:
        elapsed = time.time() - start_time
        print(f"FPS: {frame_count/elapsed:.2f}")



0: 480x640 12 cars, 3 buss, 56.7ms
Speed: 2.0ms preprocess, 56.7ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)


[msmpeg4v1 @ 0x17a459110] ext header missing, 6 left



0: 480x640 14 cars, 4 buss, 57.3ms
Speed: 1.2ms preprocess, 57.3ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 12 cars, 4 buss, 1 truck, 69.2ms
Speed: 1.0ms preprocess, 69.2ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 14 cars, 3 buss, 62.0ms
Speed: 0.9ms preprocess, 62.0ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 12 cars, 3 buss, 60.3ms
Speed: 0.8ms preprocess, 60.3ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 3 buss, 63.4ms
Speed: 1.1ms preprocess, 63.4ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 3 buss, 64.5ms
Speed: 0.9ms preprocess, 64.5ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 12 cars, 3 buss, 61.0ms
Speed: 1.0ms preprocess, 61.0ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 14 cars, 3 buss, 61.5ms
Speed: 1.1ms preprocess,

In [12]:
cap.release()
writer.release()
cv2.destroyAllWindows()
print("Done!")


Done!
